# N115 · 博弈DP与得分差

**目标：** 在双方最优前提下使用当前玩家视角。

**先修：** N098, N107, N062。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** minimax；零和；先后手；得分差；缓存；与概率决策区别。

## 从问题到算法

零和博弈DP要把双方目标放在同一个状态中。保存“当前行动者最多领先对手多少分”，取走一段收益后减去对手能取得的得分差，就自动实现max/min交替。Stone Game II还要保留会改变未来选择范围的M，不能只记录剩余位置。

## 最小实现

**本章接口：** `predict_the_winner(nums)`、`stone_game_ii(piles)`、`stone_game_iii(stone_value)`

In [1]:
from functools import cache

def predict_the_winner(nums):
    if not nums: return True
    dp=list(nums)
    for length in range(2,len(nums)+1):
        for l in range(len(nums)-length+1):
            r=l+length-1; dp[l]=max(nums[l]-dp[l+1],nums[r]-dp[l])
    return dp[0]>=0

def stone_game_ii(piles):
    n=len(piles); suffix=[0]*(n+1)
    for i in range(n-1,-1,-1): suffix[i]=suffix[i+1]+piles[i]
    @cache
    def f(i,m):
        if i==n: return 0
        if i+2*m>=n: return suffix[i]
        return max(suffix[i]-f(i+x,max(m,x)) for x in range(1,2*m+1))
    return f(0,1)

def stone_game_iii(stone_value):
    n=len(stone_value); dp=[0]*(n+1)
    for i in range(n-1,-1,-1):
        taken=0; dp[i]=float('-inf')
        for j in range(i,min(n,i+3)):
            taken+=stone_value[j]; dp[i]=max(dp[i],taken-dp[j+1])
    return 'Alice' if dp[0]>0 else 'Bob' if dp[0]<0 else 'Tie'

## 正确性、前提与复杂度

每个子问题重置为轮到当前行动者，所以下一状态的优势属于对手，需要减去。边界为空时差为0。Stone Game II用剩余总和减去对手最多所得等价于自身最多所得，且所有石子非负时可全取就最优。

**代价：** 两端博弈O(n²)时间O(n)空间；Stone III O(n)时间空间；Stone II保守O(n³)时间O(n²)缓存，递归深度O(n)。II要求非负石堆；III允许负数；平局按接口分别返回True或Tie。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,2,3,7]; dp=[0]*(len(a)+1); rows=[]
for i in range(len(a)-1,-1,-1):
    options=[sum(a[i:i+k])-dp[i+k] for k in range(1,min(3,len(a)-i)+1)]; dp[i]=max(options); rows.append((i,options,dp[i]))
show_table(['位置','取1..3项得分差','最优差'],rows)

位置,取1..3项得分差,最优差
3,[7],7
2,"[-4, 10]",10
1,"[-8, -2, 12]",12
0,"[-11, -7, -1]",-1


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert not predict_the_winner([1,5,2]) and predict_the_winner([1,5,233,7])
assert stone_game_ii([2,7,9,4,4])==10
assert stone_game_iii([1,2,3,7])=='Bob' and stone_game_iii([1,2,3,6])=='Tie'
from itertools import product
@cache
def ends(a):
    if not a: return 0
    return max(a[0]-ends(a[1:]),a[-1]-ends(a[:-1]))
@cache
def front(a):
    if not a: return 0
    return max(sum(a[:k])-front(a[k:]) for k in range(1,min(3,len(a))+1))
for a in product((-2,0,3),repeat=6):
    assert predict_the_winner(a)==(ends(a)>=0)
    d=front(a); assert stone_game_iii(a)==('Alice' if d>0 else 'Bob' if d<0 else 'Tie')
@cache
def ii_ref(a,m):
    if not a: return 0
    return max(sum(a[:k])+(sum(a[k:])-ii_ref(a[k:],max(m,k))) for k in range(1,min(len(a),2*m)+1))
for a in product((1,2),repeat=6): assert stone_game_ii(a)==ii_ref(a,1)
print('N115: 所有本章断言通过')

N115: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 推导difference=current_gain-next_difference。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 解释一类题的结论不能替代通用DP。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 486. Predict the Winner（canonical）
- 877. Stone Game（canonical）
- 1140. Stone Game II（canonical）
- 1406. Stone Game III（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。